# Homework 5b — OpenTelemetry & dlt

This notebook documents my solution to the **Module 5b (DLT workshop) homework of [LLM Zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/workshops/dlt)** by DataTalksClub.

## What's covered
Instead of manual instrumentation, this assignment uses [Pydantic Logfire](https://pydantic.dev/) for industry-standard observability, then pipes the captured traces into DuckDB using [dlt](https://dlthub.com/) for querying — covering the full observability pipeline from instrument to store to analyze.

## Stack used

| Component | Tool |
|---|---|
| Agent framework | Pydantic AI |
| Observability | Pydantic Logfire |
| Data pipeline | dlt (Python → DuckDB) |
| Query engine | DuckDB |
| LLM provider | Groq (`openai/gpt-oss-120b`) |
| Index | `minsearch.Index` (FAQ) |

> API keys are loaded from `.env` via `python-dotenv`.

## Download resources

In [1]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/workshops/dlt/homework"

!wget -nc $PREFIX/agent.py
!wget -nc $PREFIX/ingest.py
!wget -nc $PREFIX/main.py
# !wget $PREFIX/.env.example -O .env

File 'agent.py' already there; not retrieving.

File 'ingest.py' already there; not retrieving.

File 'main.py' already there; not retrieving.



## Setup — OpenAI-compatible Client

The agent uses Groq via the OpenAI-compatible API through Pydantic AI's model abstraction.

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from pathlib import Path

path = Path.cwd().parent/".env"
print(load_dotenv(path))

llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
    )

True


## Agent Setup

The FAQ agent uses Pydantic AI with a custom OpenAI provider routing to Groq, and a `search` tool backed by the minsearch index.

In [3]:
import os
from dataclasses import dataclass
from minsearch import Index
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

# Build a custom OpenAI provider routing to Groq
groq_via_openai = OpenAIResponsesModel(
    model_name="openai/gpt-oss-120b",
    provider=OpenAIProvider(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.environ.get("GROQ_API_KEY"),
    ),
)

@dataclass
class SearchDeps:
    index: Index


faq_agent = Agent(
    groq_via_openai,
    deps_type=SearchDeps,
    instructions=INSTRUCTIONS,
)

@faq_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> str:
    """Search the FAQ database for entries matching the given query."""
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}
    results = ctx.deps.index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    return results

### Test the agent

In [4]:
from ingest import build_index, load_faq_data

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

question = 'I just discovered the course. Can I join it?'
result = await faq_agent.run(question, deps=deps) # run_sync() not working in jupyter nb

print(result.output)

**Answer**

Yes—you can join the LLM Zoomcamp even if you’ve just discovered it.  

- **Enrollment:** You can start working on the material right away. The videos, notebooks, and GitHub repository are publicly available.  
- **Certificate:** If you want to earn a certificate, you’ll need to submit your capstone project **while the cohort is still accepting submissions**. The project and the required peer reviews are the only mandatory items for certification; the weekly homework is optional.

Feel free to explore the course docs, the logistics page, and the GitHub repo to get started.  

---

Is there anything else about the course (e.g., schedule, cohort details, technical setup) that you’d like to know?


Modify `agent.py` to use qroq openai sdk as above (groq_via_openai)

## Q1 — Instrument the agent with Logfire

Instrument the Pydantic AI agent with Logfire to capture traces automatically. Run the agent and observe how many spans are produced for a single question.

In [5]:
from dotenv import load_dotenv
load_dotenv()

import logfire
logfire.configure()
logfire.instrument_pydantic_ai()

from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

question = 'How do I run Ollama locally?'
result = await faq_agent.run(question, deps=deps)
# print(result.output)

Logfire project URL: https://logfire-us.pydantic.dev/jmosleem/starter-project

02:25:54.847 faq_agent run
02:25:54.848   chat openai/gpt-oss-120b
02:25:55.778   running 1 tool
02:25:55.779     running tool: search
02:25:55.792   chat openai/gpt-oss-120b


Q1) How many spans does a single agent run produce?

<font color="green">Answer: 5 </font>

## Q2 — Load traces into DuckDB with dlt

Use dlt to extract trace data from Logfire's query API and load it into a local DuckDB database. The schema is auto-discovered by dlt from the JSON structure.

In [ ]:
import os
import dlt
from dlt.sources.helpers import requests

HEADERS = {"Authorization": f"Bearer {os.getenv('LOGFIRE_READ_TOKEN')}",
           "Accept": "application/json",
           }
BASE_URL = "https://logfire-us.pydantic.dev/v1/query"


@dlt.source
def logfire_source():

    @dlt.resource(write_disposition="append", name="records")
    def records():
        params = {"sql": "SELECT * FROM records"}
        response = requests.get(BASE_URL, headers=HEADERS, params=params)
        response.raise_for_status()
        yield response.json()

    return records


pipeline = dlt.pipeline(
    pipeline_name="logfire_pipeline_q2",
    destination="duckdb",
    dataset_name="agent_traces",
)

info = pipeline.run(logfire_source())
print(info)

2026-07-21 02:25:59,185|[WARNING]|2330|8484039744|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'records__columns__values' did not receive any data during this load and therefore could not have their types inferred:
  - model_request_parameters__output_object

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'model_request_parameters__output_object': {'data_type': 'text'}})

2026-07-21 02:25:59,186|[WARNING]|2330|8484039744|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'records__columns__values__model_request_parameters__function_tools' did not receive any data during this load and therefore could not have their types inferred:
  - outer_typed_dict_key

Unless type hints are provided, these columns w

Pipeline logfire_pipeline_q2 load step completed in 0.79 seconds
1 load package(s) were loaded to destination duckdb and into dataset agent_traces
The duckdb destination used duckdb:////Users/jusnaini/Documents/MyProject/DATA-ENGINEERING/llm-zoomcamp/llm-zoomcamp-homework/05-monitoring/logfire_pipeline_q2.duckdb location to store data
Load package 1784571958.15343 is LOADED and contains no failed jobs


In [7]:
import duckdb
conn = duckdb.connect("logfire_pipeline_q2.duckdb")
print(conn.sql("SELECT * FROM information_schema.tables WHERE table_schema = 'agent_traces'"))

┌─────────────────────┬──────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│    table_catalog    │ table_schema │                                              table_name                                              │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│       varchar       │   varchar    │                                               varchar                                                │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      

Q2) How many tables did dlt create? Check with:

<font color="green">Answer: ~24 </font>

dlt creates approximately 21-24 tables in DuckDB, including the main `records` table, nested column tables, and dlt metadata tables (`_dlt_loads`, `_dlt_pipeline_state`, `_dlt_version`).

## Q3 — Query traces with an agent

Run the agent again (with Logfire instrumentation), sync the new traces into a separate DuckDB database via dlt, then query the stored token usage.

### Step 1 — Trigger the agent (generate the trace)

In [1]:
import os
import logfire
import dlt
import duckdb
from dlt.sources.helpers import requests
from dotenv import load_dotenv
load_dotenv()

# ==========================================
# STEP 1: TRIGGER THE AGENT (Generate the Trace)
# ==========================================
# 1. Initialize Logfire instrumentation
logfire.configure(token=os.getenv("LOGFIRE_TOKEN"))
logfire.instrument_pydantic_ai()

# 2. Trigger agent with traces
from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

question = "How do I run Ollama locally?"
result = await faq_agent.run(question, deps=deps)  
print(f"Agent Finished! Response: {result.output[:50]}...\n")



Logfire project URL: https://logfire-us.pydantic.dev/jmosleem/starter-project

03:32:37.779 faq_agent run
03:32:37.780   chat openai/gpt-oss-120b
03:32:39.099   running 1 tool
03:32:39.100     running tool: search
03:32:39.109   chat openai/gpt-oss-120b
Agent Finished! Response: Here’s how you can get Ollama up and running on yo...



### Step 2 — Run dlt to extract the fresh spans

In [2]:
# ==========================================
# STEP 2: RUN DLT (Extract the Fresh Spans)
# ==========================================
HEADERS = {
    "Authorization": f"Bearer {os.getenv('LOGFIRE_READ_TOKEN')}",
    "Accept": "application/json",
}
BASE_URL = "https://logfire-us.pydantic.dev/v1/query"

@dlt.source
def logfire_source():
    @dlt.resource(write_disposition="replace", name="records")
    def records():
        # Pull the records we just generated!
        params = {"sql": "SELECT * FROM records"}
        response = requests.get(BASE_URL, headers=HEADERS, params=params)
        response.raise_for_status()
        yield response.json()
    return records

pipeline = dlt.pipeline(
    pipeline_name="logfire_pipeline_q3",
    destination="duckdb",
    dataset_name="agent_traces",
)

# Run the pipeline to populate DuckDB
info = pipeline.run(logfire_source())
print("DLT Sync Finished!\n")



2026-07-21 03:33:46,881|[WARNING]|51841|8484039744|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'records__columns__values' did not receive any data during this load and therefore could not have their types inferred:
  - model_request_parameters__output_object

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'model_request_parameters__output_object': {'data_type': 'text'}})

2026-07-21 03:33:46,882|[WARNING]|51841|8484039744|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'records__columns__values__model_request_parameters__function_tools' did not receive any data during this load and therefore could not have their types inferred:
  - outer_typed_dict_key

Unless type hints are provided, these columns

DLT Sync Finished!



In [4]:
# ==========================================
# STEP 3: QUERY THE TOKENS (Get your Answer)
# ==========================================
conn = duckdb.connect("logfire_pipeline_q3.duckdb")

query_result = conn.sql("""
    SELECT SUM(gen_ai_usage_input_tokens) AS total_input_tokens
    FROM logfire_pipeline_q3.agent_traces.records__columns__values 
    WHERE logfire_metrics__gen_ai_client_token_usage__total IS NOT NULL
    AND agent_name = 'faq_agent'
    ;
""")

print("--- Question 3 Final Token Count ---")
print(query_result)

conn.close()

--- Question 3 Final Token Count ---
┌────────────────────┐
│ total_input_tokens │
│       int128       │
├────────────────────┤
│              30736 │
└────────────────────┘



Q3) Find the input token usage for the agent run from Q1.

<font color="green"> Answer: 10000 - 20000 </font>

In [6]:
import pandas as pd 

conn = duckdb.connect("logfire_pipeline_q3.duckdb")
df = pd.read_sql_query("SELECT * FROM agent_traces.records__columns__values", conn)
conn.close()
df

/var/folders/yz/01283wk557g3r_2q2cbpz5dr0000gn/T/ipykernel_51841/3196618092.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM agent_traces.records__columns__values", conn)


,value,_dlt_parent_id,_dlt_list_idx,_dlt_id,value__v_text,value__v_bool,type,properties__tool_response__type,properties__tool_arguments__type,properties__final_result__type,...,process_pid,process_runtime_description,process_runtime_name,process_runtime_version,service_instance_id,service_name,service_version,telemetry_sdk_language,telemetry_sdk_name,telemetry_sdk_version
0,2026-07-20 19:32:42.408511+00:00,6cPGPMzf5HU4aA,0,VVm/W8QThYwNDQ,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-07-20 19:32:42.408511+00:00,6cPGPMzf5HU4aA,1,10zcIo+ggJA9Mg,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-07-20 13:08:14.586665+00:00,6cPGPMzf5HU4aA,2,PAjFvCm7aFvauQ,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-07-20 13:08:45.974044+00:00,6cPGPMzf5HU4aA,3,Alp05HL6HjPeJQ,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-07-20 13:08:14.586665+00:00,6cPGPMzf5HU4aA,4,X7RJv0nlOWgf/Q,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4395,2026-07-20 00:00:00+00:00,WaeT1beiKe81BA,95,vyG4uAq00sPnzA,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4396,2026-07-20 00:00:00+00:00,WaeT1beiKe81BA,96,Q7Q4PtJf7A7pqA,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4397,2026-07-20 00:00:00+00:00,WaeT1beiKe81BA,97,mLIvuGogI/6auA,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4398,2026-07-20 00:00:00+00:00,WaeT1beiKe81BA,98,E1TYsnkKCoUlYA,NaN,None,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
